# FullStack IA · Fondation d'ingénierie en IA

**Objectif :** Python solide, programmation orientée objet, Git et GitHub.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Classes et objets

Tu n'écriras pas beaucoup de classes au début, mais tu en **utiliseras** tout le temps : un modèle scikit-learn, un DataFrame pandas, un tokenizer Hugging Face sont des objets. Comprendre les classes, c'est comprendre leur documentation.

Une **classe** est un plan de fabrication. Un **objet** est un exemplaire fabriqué à partir de ce plan. Il a des **attributs** (ses données) et des **méthodes** (ses fonctions).

In [ ]:
class Compteur:
    def __init__(self, nom):
        self.nom = nom          # attribut
        self.valeur = 0

    def incrementer(self, pas=1):   # méthode
        self.valeur += pas
        return self

c = Compteur("lecons")
c.incrementer()
c.incrementer(3)
print(c.nom, c.valeur)

`__init__` est appelé à la création de l'objet. `self` désigne l'objet lui-même : c'est toujours le premier paramètre d'une méthode.

## Le modèle fit / predict

Tous les modèles de scikit-learn suivent la même interface : `fit` apprend à partir des données, `predict` prédit. Voici un modèle minimal qui prédit toujours la moyenne. C'est ce qu'on appelle une **baseline**.

In [ ]:
class ModeleMoyenne:
    def fit(self, y):
        self.moyenne_ = sum(y) / len(y)    # le _ final : attribut appris
        return self

    def predict(self, n):
        return [self.moyenne_] * n

m = ModeleMoyenne().fit([3, 5, 10])
print(m.predict(2))

> **À retenir** : dans scikit-learn, les attributs appris pendant `fit` finissent par un tiret bas, comme `coef_` ou `intercept_`. Tu les reconnaîtras en semaine 4.

### Exercice 1 · Un modèle médiane

Écris une classe `ModeleMediane` avec une méthode `fit(self, y)` qui stocke la médiane de `y` dans `self.mediane_` et renvoie `self`, et une méthode `predict(self, n)` qui renvoie une liste de `n` fois cette médiane. Médiane : trie les valeurs ; si leur nombre est impair, prends celle du milieu, sinon la moyenne des deux du milieu.

<details><summary>Indice</summary>

len(s) // 2 donne l'indice du milieu. Pour une taille paire, moyenne de s[k-1] et s[k].
</details>

In [ ]:
class ModeleMediane:
    def fit(self, y):
        pass

    def predict(self, n):
        pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
m = ModeleMediane()
assert m.fit([5, 1, 9]) is m, "fit doit renvoyer self."
assert m.mediane_ == 5, f"Pour [5, 1, 9], mediane_ vaut {m.mediane_}, j'attendais 5."
assert m.predict(2) == [5, 5], "predict(2) doit renvoyer [5, 5]."
assert ModeleMediane().fit([4, 1, 3, 10]).mediane_ == 3.5, "Pour [4, 1, 3, 10], la médiane est 3.5."
print("Bravo, tout est correct.")

---
## Programmation orientée objet

Chez Jedha, le module « Bonnes pratiques Python » consacre deux cours à la POO, et la leçon « Pourquoi la POO est importante pour l'IA » explique pourquoi : tous les outils que tu vas utiliser (scikit-learn, PyTorch, Hugging Face, LangChain) sont construits avec des classes. Savoir lire et écrire une classe propre, c'est savoir lire leur documentation et leur code source.

*Animation interactive « oop » : à voir sur la plateforme web.*

## Une classe, des objets

Une **classe** décrit un type d'objet : ses **attributs** (ses données) et ses **méthodes** (ce qu'il sait faire). Chaque **objet** créé à partir de la classe a ses propres valeurs.

In [ ]:
class Article:
    tva = 0.20                       # attribut de classe : partagé par tous les articles

    def __init__(self, nom, prix_ht, quantite=1):
        self.nom = nom               # attributs d'instance : propres à chaque objet
        self.prix_ht = prix_ht
        self.quantite = quantite

    def total_ttc(self):
        return round(self.prix_ht * self.quantite * (1 + self.tva), 2)

stylo = Article("Stylo", 1.5, 10)
cahier = Article("Cahier", 3.2)
print(stylo.total_ttc(), cahier.total_ttc())
print(isinstance(stylo, Article), type(cahier).__name__)

## Les méthodes spéciales (dunder)

Les méthodes entourées de doubles tirets bas, comme `__init__`, permettent à tes objets de se comporter comme des objets Python natifs : s'afficher, se comparer, avoir une longueur, s'additionner.

In [ ]:
class Panier:
    def __init__(self):
        self.articles = []

    def ajouter(self, article):
        self.articles.append(article)
        return self                          # permet d'enchaîner les appels

    def __len__(self):                       # len(panier)
        return sum(a.quantite for a in self.articles)

    def __repr__(self):                      # affichage dans la console
        return f"Panier({len(self.articles)} lignes, {len(self)} articles)"

    def __iter__(self):                      # for article in panier
        return iter(self.articles)

p = Panier().ajouter(stylo).ajouter(cahier)
print(len(p))
for a in p:
    print("-", a.nom)
p

C'est exactement ce qui se passe quand tu fais `len(df)` sur un DataFrame ou `for batch in dataloader` en PyTorch : ces classes définissent `__len__` et `__iter__`.

## Héritage

Une classe peut **hériter** d'une autre : elle récupère tout son comportement et peut en modifier une partie. `super()` appelle la version de la classe parente.

In [ ]:
class ArticleSolde(Article):
    def __init__(self, nom, prix_ht, quantite=1, remise=0.3):
        super().__init__(nom, prix_ht, quantite)
        self.remise = remise

    def total_ttc(self):                     # on redéfinit (override) la méthode
        return round(super().total_ttc() * (1 - self.remise), 2)

sac = ArticleSolde("Sac", 40, remise=0.25)
print(sac.total_ttc(), isinstance(sac, Article))

Le code qui manipule des `Article` fonctionne aussi avec des `ArticleSolde` sans rien changer : c'est le **polymorphisme**. scikit-learn en est l'exemple parfait : `fit` et `predict` existent sur tous les modèles, quelle que soit leur classe.

## Composition plutôt qu'héritage

L'héritage exprime « est un » (un `ArticleSolde` est un `Article`). La **composition** exprime « a un » : un `Panier` a des articles. En pratique, on compose bien plus souvent qu'on hérite. Une `Pipeline` scikit-learn, par exemple, *contient* des étapes.

> **À retenir** : une classe = des données + les fonctions qui les manipulent. Hérite pour spécialiser un comportement, compose pour assembler des briques.

### Exercice 1 · Un compte bancaire

Écris une classe `Compte` avec : `__init__(self, titulaire, solde=0)`, une méthode `deposer(montant)`, une méthode `retirer(montant)` qui lève une `ValueError` si le solde est insuffisant, et `__repr__` qui renvoie exactement `Compte(Sam, 120.0 €)` pour un compte de Sam avec 120 euros.

<details><summary>Indice</summary>

raise ValueError("...") lève une erreur. Dans __repr__, utilise une f-string avec self.titulaire et self.solde.
</details>

In [ ]:
class Compte:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
c = Compte("Sam", 100)
c.deposer(20)
assert repr(c) == "Compte(Sam, 120.0 €)", f"repr(c) vaut {repr(c)!r}."
try:
    c.retirer(500)
    assert False, "retirer(500) aurait dû lever une ValueError."
except ValueError:
    pass
c.retirer(20)
assert c.solde == 100, "Après retirer(20), le solde devrait être 100."
print("Bravo, tout est correct.")

### Exercice 2 · Un dataset itérable

Écris une classe `MiniDataset` qui reçoit une liste `X` et une liste `y` de même longueur. `len(ds)` doit renvoyer le nombre d'exemples, `ds[i]` le tuple `(X[i], y[i])` (méthode `__getitem__`), et le constructeur doit lever une `ValueError` si les longueurs diffèrent. C'est exactement l'interface d'un `Dataset` PyTorch.

<details><summary>Indice</summary>

Trois méthodes : __init__, __len__ et __getitem__(self, i). Avec __getitem__, la boucle for fonctionne aussi.
</details>

In [ ]:
class MiniDataset:
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ds = MiniDataset([[1, 2], [3, 4], [5, 6]], [0, 1, 0])
assert len(ds) == 3, "len(ds) devrait valoir 3."
assert ds[1] == ([3, 4], 1), f"ds[1] vaut {ds[1]!r}."
assert [c for _, c in ds] == [0, 1, 0], "On doit pouvoir parcourir le dataset avec for."
try:
    MiniDataset([1, 2], [1])
    assert False, "Des longueurs différentes doivent lever une ValueError."
except ValueError:
    pass
print("Bravo, tout est correct.")

### Exercice 3 · Polymorphisme : des formes

Crée une classe `Forme` avec une méthode `aire()` qui lève `NotImplementedError`, puis deux classes filles `Rectangle(largeur, hauteur)` et `Cercle(rayon)` qui la redéfinissent (utilise `math.pi`). Enfin, crée `aire_totale`, la somme des aires des formes de la liste `formes`.

<details><summary>Indice</summary>

class Rectangle(Forme): ... La somme se fait sans tester le type de chaque forme : c'est le polymorphisme.
</details>

In [ ]:
import math

# tes classes ici

formes = [Rectangle(2, 3), Cercle(1), Rectangle(1, 1)]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import math
assert issubclass(Rectangle, Forme) and issubclass(Cercle, Forme), "Rectangle et Cercle doivent hériter de Forme."
try:
    Forme().aire()
    assert False, "Forme().aire() doit lever NotImplementedError."
except NotImplementedError:
    pass
assert abs(aire_totale - (7 + math.pi)) < 1e-9, f"aire_totale vaut {aire_totale}."
print("Bravo, tout est correct.")

---
## POO avancée : dataclasses, propriétés, classes abstraites

Ces outils rendent ton code plus court, plus sûr et plus lisible. Tu les retrouveras partout dans le code professionnel, notamment dans FastAPI et Pydantic, qui reposent sur les annotations de type.

## Les annotations de type

Elles indiquent le type attendu des paramètres et du retour. Python ne les vérifie pas à l'exécution, mais ton éditeur et les outils comme `mypy` s'en servent pour détecter les erreurs.

In [ ]:
def moyenne_ponderee(valeurs: list[float], poids: list[float]) -> float:
    return sum(v * p for v, p in zip(valeurs, poids)) / sum(poids)

moyenne_ponderee([12, 15, 9], [1, 2, 1])

## Les dataclasses

`@dataclass` écrit pour toi `__init__`, `__repr__` et `__eq__` à partir des attributs annotés.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Experience:
    nom: str
    taux_apprentissage: float = 0.01
    epoques: int = 10
    tags: list[str] = field(default_factory=list)   # jamais de liste mutable comme valeur par défaut directe

e1 = Experience("baseline")
e2 = Experience("rapide", taux_apprentissage=0.1, tags=["test"])
print(e1)
print(e1 == Experience("baseline"))

`@dataclass(frozen=True)` rend l'objet immuable : pratique pour une configuration qui ne doit pas changer.

## Les propriétés

Une **propriété** ressemble à un attribut mais exécute du code : pour calculer une valeur à la volée, ou pour **valider** une affectation.

In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self.celsius = celsius            # passe par le setter ci-dessous

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, valeur: float):
        if valeur < -273.15:
            raise ValueError("En dessous du zéro absolu !")
        self._celsius = valeur

    @property
    def fahrenheit(self) -> float:        # propriété calculée, en lecture seule
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print(t.fahrenheit)
try:
    t.celsius = -300
except ValueError as err:
    print("Refusé :", err)

## Méthodes de classe et méthodes statiques

In [ ]:
class Modele:
    def __init__(self, params: dict):
        self.params = params

    @classmethod
    def depuis_json(cls, texte: str) -> "Modele":     # constructeur alternatif
        import json
        return cls(json.loads(texte))

    @staticmethod
    def version() -> str:                             # n'utilise ni self ni cls
        return "1.0"

m = Modele.depuis_json('{"profondeur": 3}')
print(m.params, Modele.version())

`AutoModel.from_pretrained(...)` chez Hugging Face est une méthode de classe de ce type.

## Les classes abstraites

Une **classe abstraite** définit un contrat : toute classe fille doit implémenter certaines méthodes, sinon on ne peut pas l'instancier.

In [ ]:
from abc import ABC, abstractmethod

class Estimateur(ABC):
    @abstractmethod
    def fit(self, X, y): ...

    @abstractmethod
    def predict(self, X): ...

    def fit_predict(self, X, y):          # méthode concrète, partagée par toutes les filles
        return self.fit(X, y).predict(X)

class Constant(Estimateur):
    def fit(self, X, y):
        self.valeur_ = max(set(y), key=y.count)
        return self

    def predict(self, X):
        return [self.valeur_] * len(X)

print(Constant().fit_predict([[1], [2], [3]], ["a", "b", "a"]))
try:
    Estimateur()
except TypeError as err:
    print("Impossible :", err)

> **À retenir** : `@dataclass` pour les objets qui portent surtout des données, `@property` pour valider ou calculer, `ABC` pour imposer une interface commune à plusieurs implémentations.

### Exercice 1 · Une commande en dataclass

Crée deux dataclasses : `Ligne(produit: str, prix: float, quantite: int = 1)` avec une propriété `total` (prix × quantité), et `Commande(client: str, lignes: list[Ligne])` (liste vide par défaut, avec `field(default_factory=list)`) avec une propriété `montant` (somme des totaux, arrondie à 2 décimales).

<details><summary>Indice</summary>

Une propriété dans une dataclass s'écrit comme dans une classe normale, avec @property.
</details>

In [ ]:
from dataclasses import dataclass, field

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import dataclasses
assert dataclasses.is_dataclass(Ligne) and dataclasses.is_dataclass(Commande), "Ligne et Commande doivent être des dataclasses."
c = Commande("Sam", [Ligne("Clavier", 49.9), Ligne("Souris", 19.95, 2)])
assert c.montant == 89.8, f"montant vaut {c.montant}, j'attendais 89.8."
a, b = Commande("A"), Commande("B")
a.lignes.append(Ligne("x", 1))
assert b.lignes == [], "Les commandes partagent la même liste : utilise field(default_factory=list)."
assert Ligne("x", 2.0, 3) == Ligne("x", 2.0, 3), "Deux lignes identiques doivent être égales (c'est le rôle de @dataclass)."
print("Bravo, tout est correct.")

### Exercice 2 · Valider avec une propriété

Écris une classe `Probabilite` dont l'attribut `valeur` est une propriété : son setter lève une `ValueError` si la valeur n'est pas entre 0 et 1. Ajoute une propriété en lecture seule `cote` qui renvoie `valeur / (1 - valeur)`.

<details><summary>Indice</summary>

Stocke la vraie valeur dans self._valeur, et passe par @property / @valeur.setter.
</details>

In [ ]:
class Probabilite:
    def __init__(self, valeur):
        self.valeur = valeur

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
p = Probabilite(0.75)
assert abs(p.cote - 3) < 1e-9, f"cote vaut {p.cote}, j'attendais 3."
for mauvais in [1.5, -0.1]:
    try:
        Probabilite(mauvais)
        assert False, f"Probabilite({mauvais}) devrait lever une ValueError."
    except ValueError:
        pass
try:
    p.valeur = 2
    assert False, "Modifier valeur avec 2 devrait aussi lever une ValueError."
except ValueError:
    pass
print("Bravo, tout est correct.")

### Exercice 3 · Ton propre estimateur

En héritant de `Estimateur` (fourni), écris `PlusProcheMoyenne` : un classifieur qui, dans `fit(X, y)`, calcule le **centre** (moyenne coordonnée par coordonnée) de chaque classe et le stocke dans `self.centres_` (un dictionnaire classe → liste), puis qui, dans `predict(X)`, renvoie pour chaque point la classe du centre le plus proche (distance euclidienne).

<details><summary>Indice</summary>

zip(*points) regroupe les coordonnées par colonne. math.dist(a, b) calcule la distance euclidienne.
</details>

In [ ]:
from abc import ABC, abstractmethod

class Estimateur(ABC):
    @abstractmethod
    def fit(self, X, y): ...
    @abstractmethod
    def predict(self, X): ...

class PlusProcheMoyenne(Estimateur):
    pass

X = [[1, 1], [1, 2], [8, 8], [9, 8]]
y = ["bas", "bas", "haut", "haut"]

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
X = [[1, 1], [1, 2], [8, 8], [9, 8]]
y = ["bas", "bas", "haut", "haut"]
m = PlusProcheMoyenne()
assert m.fit(X, y) is m, "fit doit renvoyer self."
assert m.centres_["bas"] == [1.0, 1.5] and m.centres_["haut"] == [8.5, 8.0], f"centres_ vaut {m.centres_}."
assert m.predict([[2, 2], [7, 9], [0, 0]]) == ["bas", "haut", "bas"], f"predict renvoie {m.predict([[2, 2], [7, 9], [0, 0]])}."
print("Bravo, tout est correct.")

---
## Git et GitHub

**Git** enregistre l'historique de ton code : chaque *commit* est une photo de ton projet à un instant donné. **GitHub** héberge ces projets en ligne. C'est aussi ton **portfolio** : les recruteurs regarderont ton GitHub.

## Les notions à connaître

- **Dépôt (repository)** : un dossier suivi par Git.
- **Commit** : un enregistrement de modifications, avec un message qui explique ce qui a changé.
- **Branche** : une ligne de développement parallèle. `main` est la branche principale.
- **Remote** : la copie du dépôt sur GitHub, souvent appelée `origin`.

## Le cycle de base

```bash
git clone https://github.com/ton-pseudo/mon-projet.git   # copier un dépôt
cd mon-projet
# ... tu modifies des fichiers ...
git status                  # qu'est-ce qui a changé ?
git add analyse.py          # préparer ce fichier pour le commit
git commit -m "Ajoute le calcul des statistiques par espèce"
git push                    # envoyer sur GitHub
git pull                    # récupérer les changements distants
```

Pour démarrer un projet local : `git init`, puis crée un dépôt vide sur GitHub et relie-le avec `git remote add origin <url>` puis `git push -u origin main`.

## Les bonnes habitudes

- Des commits **petits et fréquents**, avec un message au présent qui dit ce que fait le commit.
- Un fichier **README.md** dans chaque projet : le problème traité, les données, comment lancer le code, les résultats.
- Un fichier **.gitignore** pour exclure ce qui ne doit pas être publié :

```bash
# .gitignore
.venv/
__pycache__/
.ipynb_checkpoints/
.env          # tes clés d'API : ne JAMAIS les publier
data/*.csv    # les gros fichiers de données
```

> **À retenir** : ne publie jamais une clé d'API ou un mot de passe sur GitHub, même dans un dépôt privé. Mets-les dans un fichier `.env` listé dans `.gitignore`.

**À faire :**

- [ ] Configurer Git : git config --global user.name et user.email
- [ ] Créer un dépôt « mise-a-niveau-ia » sur GitHub avec un README
- [ ] Le cloner sur ton ordinateur avec git clone
- [ ] Ajouter un fichier, faire un commit et un push
- [ ] Vérifier sur github.com que le fichier apparaît

---
## Markdown et README qui donnent envie

Le **Markdown** est le format de texte des README, des notebooks Jupyter, de la documentation et de… cette plateforme. Un recruteur passe en moyenne moins d'une minute sur un dépôt : le README est ta vitrine.

## La syntaxe essentielle

```markdown
# Titre de niveau 1
## Titre de niveau 2

Du texte en **gras**, en *italique*, et du `code` en ligne.

- une liste à puces
- un deuxième point

1. une liste numérotée
2. deuxième étape

[un lien](https://www.jedha.co)   ![une image](images/courbe.png)

| Modèle | Exactitude |
| --- | --- |
| Baseline | 0.62 |
| Régression logistique | 0.80 |
```

Pour du code sur plusieurs lignes, on l'entoure de trois accents graves suivis du nom du langage.

## La structure d'un bon README de projet

1. **Un titre et une phrase** qui disent ce que fait le projet et pour qui.
2. **Un visuel** : capture d'écran, graphique clé ou GIF de démo.
3. **Les résultats** en chiffres, comparés à une baseline.
4. **Comment le lancer** : installation et commande, copiables-collables.
5. **Les données** : source, taille, licence.
6. **La démarche** : les choix importants et leurs raisons.
7. **Les limites et les pistes d'amélioration** : c'est là que tu montres ta maturité.

> **À retenir** : écris tes README en anglais, au présent, et mets les résultats en haut. Un recruteur doit comprendre en 30 secondes ce que tu as fait et à quel point ça marche.

**À faire :**

- [ ] Ajouter un README complet à ton dépôt « mise-a-niveau-ia » en suivant les 7 points
- [ ] Créer ton README de profil GitHub (un dépôt qui porte ton nom d'utilisateur)
- [ ] Épingler tes 3 meilleurs dépôts sur ton profil